# Дообучение NER на извлечение товаров и брендов (русский, ОФД)

Это главный новый обучаемый компонент. Дообучаем xlm-roberta-base на
token-classification: по тексту чековой позиции размечать слова тегами
B-GOOD/I-GOOD/B-BRAND/I-BRAND/O.

xlm-roberta мультиязычна по конструкции, поэтому спокойно работает со смешанным
русско-латинским текстом чеков (бренды латиницей внутри русских позиций). Берём
готовую предобученную модель и дообучаем под нашу разметку.

Данные подготовлены в ноутбуке 07: 19077 train / 2384 val / 2386 test, в формате
JSONL со списками слов и тегов. Метки даны на уровне слов; здесь выровняем их по
сабтокенам токенизатора.

Метрику считаем не по accuracy (она завышена из-за доминирования тега O), а по F1
на сущностях через seqeval, отдельно по товарам и брендам, плюс общую метрику
соревнования ОФД (F1_good + 2·F1_brand)/3.

In [1]:
import torch
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))

GPU доступен: True
Устройство: Tesla T4


## Установка и монтирование Drive

Ставим transformers, datasets и seqeval (для метрик NER). Монтируем Google Drive,
где лежат подготовленные данные и куда будем сохранять веса.

In [2]:
!pip install transformers datasets seqeval -q

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
data_dir = Path("/content/drive/MyDrive/receipt-ai/ofd_ner")
save_dir = Path("/content/drive/MyDrive/receipt-ai/ner_ru")
save_dir.mkdir(parents=True, exist_ok=True)

for split in ["train", "validation", "test"]:
    p = data_dir / f"{split}.jsonl"
    print(f"{split}: {'есть' if p.exists() else 'НЕТ'} — {p}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
train: есть — /content/drive/MyDrive/receipt-ai/ofd_ner/train.jsonl
validation: есть — /content/drive/MyDrive/receipt-ai/ofd_ner/validation.jsonl
test: есть — /content/drive/MyDrive/receipt-ai/ofd_ner/test.jsonl


## Загрузка данных

Читаем три JSONL через datasets. В каждой записи список слов (tokens) и список
строковых тегов (ner_tags). Переводим строковые теги в числовые индексы - это
нужно модели. Порядок меток фиксируем явно, чтобы он совпадал с инференсом.

In [3]:
from datasets import load_dataset

LABELS = ["O", "B-GOOD", "I-GOOD", "B-BRAND", "I-BRAND"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}

dataset = load_dataset("json", data_files={
    "train": str(data_dir / "train.jsonl"),
    "validation": str(data_dir / "validation.jsonl"),
    "test": str(data_dir / "test.jsonl"),
})

def encode_tags(example):
    example["labels"] = [label2id[t] for t in example["ner_tags"]]
    return example

dataset = dataset.map(encode_tags)

print(dataset)
print("\nПример:")
print("tokens:", dataset["train"][0]["tokens"])
print("ner_tags:", dataset["train"][0]["ner_tags"])
print("labels:", dataset["train"][0]["labels"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/19077 [00:00<?, ? examples/s]

Map:   0%|          | 0/2384 [00:00<?, ? examples/s]

Map:   0%|          | 0/2386 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'labels'],
        num_rows: 19077
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'labels'],
        num_rows: 2384
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'labels'],
        num_rows: 2386
    })
})

Пример:
tokens: ['Контейнер', 'стеклянный', 'прямоугольный', 'SNIPS', '1', ',', '5', 'л', ',', '16', ',', '5', 'х', '21', 'х', '8', ',', '8', 'см', ',', 'зе']
ner_tags: ['B-GOOD', 'O', 'O', 'B-BRAND', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
labels: [1, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## Токенизация и выравнивание меток

Метки у нас стоят на словах, а xlm-roberta бьёт текст на сабтокены - одно слово
может стать несколькими кусочками. Например «Контейнер» может разбиться на
«Контей» + «нер». Нужно решить, какие сабтокены получают метку.

Стандартный приём: метку ставим только на первый сабтокен слова, остальные
сабтокены этого слова получают -100. Значение -100 функция потерь игнорирует,
так модель учится предсказывать метку один раз на слово, а не на каждый кусочек.
Спецтокены (начало/конец последовательности) тоже получают -100.

Передаём токенизатору уже разбитый на слова текст (is_split_into_words=True) и
через word_ids() узнаём, какому слову принадлежит каждый сабтокен.

In [5]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 64

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=max_length,
        is_split_into_words=True,
    )
    all_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev:
                label_ids.append(labels[wid])
            else:
                label_ids.append(-100)
            prev = wid
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

ex = tokenized_ds["train"][0]
toks = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print("Проверка выравнивания (сабтокен -> метка):")
for t, l in zip(toks, ex["labels"]):
    lbl = id2label[l] if l != -100 else "—(-100)"
    print(f"  {t:20s} {lbl}")

Map:   0%|          | 0/2384 [00:00<?, ? examples/s]

Проверка выравнивания (сабтокен -> метка):
  <s>                  —(-100)
  ▁Кон                 B-GOOD
  тей                  —(-100)
  нер                  —(-100)
  ▁стек                O
  ля                   —(-100)
  нный                 —(-100)
  ▁прямо               O
  у                    —(-100)
  гол                  —(-100)
  ьный                 —(-100)
  ▁S                   B-BRAND
  NI                   —(-100)
  PS                   —(-100)
  ▁1                   O
  ▁                    O
  ,                    —(-100)
  ▁5                   O
  ▁л                   O
  ▁                    O
  ,                    —(-100)
  ▁16                  O
  ▁                    O
  ,                    —(-100)
  ▁5                   O
  ▁х                   O
  ▁21                  O
  ▁х                   O
  ▁8                   O
  ▁                    O
  ,                    —(-100)
  ▁8                   O
  ▁см                  O
  ▁                    O
  ,         

## Модель, коллатор и метрики

Загружаем xlm-roberta-base с头 головой token-classification на 5 классов. Голова
инициализируется случайно, её и дообучаем под нашу задачу.

Коллатор DataCollatorForTokenClassification сам паддит и тексты, и метки в батче
(метки паддятся значением -100, чтобы паддинг не влиял на потери).

Метрики считаем через seqeval по сущностям, а не по отдельным токенам. seqeval
собирает из BIO-тегов целые сущности и сравнивает их. Выводим F1 отдельно по GOOD и BRAND, и общую метрику соревнования ОФД
(F1_good + 2·F1_brand)/3, чтобы наши числа были сопоставимы с лидербордом.
При подсчёте выбрасываем позиции с меткой -100.

In [7]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [8]:
import numpy as np
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

collator = DataCollatorForTokenClassification(tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred, label in zip(preds, labels):
        tp, tl = [], []
        for p, l in zip(pred, label):
            if l != -100:
                tp.append(id2label[p])
                tl.append(id2label[l])
        true_preds.append(tp)
        true_labels.append(tl)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    f1_good = results.get("GOOD", {}).get("f1", 0.0)
    f1_brand = results.get("BRAND", {}).get("f1", 0.0)
    ofd_metric = (f1_good + 2 * f1_brand) / 3

    return {
        "f1_good": f1_good,
        "f1_brand": f1_brand,
        "ofd_metric": ofd_metric,
        "overall_f1": results["overall_f1"],
    }

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Настройка обучения

Дообучаем уже готовую модель, поэтому скорость обучения небольшая. Эпох
немного т.к. NER на xlm-roberta сходится быстро, начнём с 4 и посмотрим на кривую.
Лучшую модель отбираем по метрике соревнования ОФД на
валидации, чтобы целиться именно в товары и бренды, а не в общий F1, где
доминирует фоновый класс.

In [9]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/receipt-ai/checkpoints/ner_ru",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="ofd_metric",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
print("готово к обучению")

готово к обучению


## Запуск обучения

После каждой эпохи Trainer показывает потери на обучении и валидации, F1 по
товарам и брендам, метрику ОФД и общий F1. Чекпойнты на Drive. В конце Trainer
сам подгрузит лучшую по метрике ОФД версию.

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Good,F1 Brand,Ofd Metric,Overall F1
1,0.119784,0.105485,0.933333,0.778528,0.830130,0.872253
2,0.091486,0.100393,0.935133,0.802318,0.846589,0.883369
3,0.052973,0.098685,0.940009,0.812224,0.854819,0.889691
4,0.053391,0.098891,0.942989,0.827318,0.865875,0.898629


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4772, training_loss=0.09254043250151872, metrics={'train_runtime': 655.0691, 'train_samples_per_second': 116.488, 'train_steps_per_second': 7.285, 'total_flos': 1490773133523960.0, 'train_loss': 0.09254043250151872, 'epoch': 4.0})

### Результаты обучения

Модель стабильно растёт все 4 эпохи без признаков переобучения: валидационные
потери почти не меняются, а метрики по сущностям продолжают
улучшаться. К четвёртой эпохе:

- F1 по товарам: 0.94
- F1 по брендам: 0.83
- метрика ОФД (F1_good + 2·F1_brand)/3: 0.87
- общий F1: 0.90

Товары извлекаются заметно лучше брендов. Товар обычно стоит в
начале позиции на предсказуемом месте и почти всегда однословный. Бренд сложнее:
он бывает в любом месте строки, часто латиницей, иногда многословный, а в трети
позиций его нет вовсе.

Рост между эпохами 3 и 4 ещё заметный, так что
модель, похоже, не уперлась в потолок и можно попробовать 5-6 эпох и посмотреть,
добавит ли это резкультата. Но и текущий результат рабочий.

## Проверка: обучение на 6 эпох

Четвёртая эпоха ещё растила бренд, потолок не достигнут. Пробуем 6 эпох, обучая
с нуля (чтобы расписание learning rate спадало корректно к концу). Сравним
финальные метрики с версией на 4 эпохах и оставим ту, что лучше по метрике ОФД.

In [12]:
args_cont = TrainingArguments(
    output_dir="/content/drive/MyDrive/receipt-ai/checkpoints/ner_ru_cont",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="ofd_metric",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
)

trainer_cont = Trainer(
    model=model,
    args=args_cont,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer_cont.train()

Epoch,Training Loss,Validation Loss,F1 Good,F1 Brand,Ofd Metric,Overall F1
1,0.034811,0.113520,0.943820,0.826429,0.865559,0.899033
2,0.025169,0.124180,0.943172,0.828297,0.866589,0.899225


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2386, training_loss=0.03269892503809549, metrics={'train_runtime': 335.7319, 'train_samples_per_second': 113.644, 'train_steps_per_second': 7.107, 'total_flos': 744924181444560.0, 'train_loss': 0.03269892503809549, 'epoch': 2.0})

## Выбор модели и оценка на тесте

Дообучение на 2 эпохи поверх (всего 6) прироста не дало: F1 по бренду остался
0.83, а валидационные потери выросли. Значит модель на 4 эпохах уже у потолка для этих данных. Берём её.

Загружаем лучший чекпойнт первого обучения и оцениваем на тестовой части,
которую модель не видела ни при обучении, ни при отборе эпохи.

In [13]:
from transformers import AutoModelForTokenClassification

ckpt_root = Path("/content/drive/MyDrive/receipt-ai/checkpoints/ner_ru")
checkpoints = sorted(ckpt_root.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
print("Найденные чекпойнты:", [c.name for c in checkpoints])

best_model = AutoModelForTokenClassification.from_pretrained(str(checkpoints[-1]))

eval_trainer = Trainer(
    model=best_model,
    args=args,
    eval_dataset=tokenized_ds["test"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

test_metrics = eval_trainer.evaluate()
print("\nМетрики на тесте:")
for k, v in test_metrics.items():
    if k.startswith("eval_"):
        print(f"  {k[5:]}: {round(v, 4)}")

Найденные чекпойнты: ['checkpoint-3579', 'checkpoint-4772']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1 Good,F1 Brand,Ofd Metric,Overall F1
No log,0.098667,0,0.951461,0.822608,0.865559,0.900935



Метрики на тесте:
  loss: 0.0987
  f1_good: 0.9515
  f1_brand: 0.8226
  ofd_metric: 0.8656
  overall_f1: 0.9009


### Финальные метрики на тесте

На тестовой части:

- F1 по товарам: 0.95
- F1 по брендам: 0.82
- метрика ОФД: 0.87
- общий F1: 0.90

Цифры на тесте практически совпали с валидационными, модель не переобучилась и
держит качество на новых данных. Товары извлекаются отлично, бренды
заметно труднее, что ожидаемо: бренд бывает где угодно в строке, часто
латиницей, в трети позиций его нет вовсе, и модель должна решать ещё и сам факт
наличия бренда.

Это нормальный результат для главного компонента. Для сравнения: официальная
метрика соревнования ОФД у победителей была выше, но они решали задачу с
доразметкой и ансамблями; наша одиночная xlm-roberta-base за 11 минут обучения
даёт 0.87, что для учебного проекта более чем достойно.

## Разбор ошибок

Общий F1 скрывает детали. Посмотрим подробный seqeval-отчёт (precision/recall по
каждому типу) и конкретные примеры, где модель ошиблась на бренде - это покажет,
что именно ей трудно: пропускает бренд, путает с товаром, или цепляет лишнее.

In [14]:
from seqeval.metrics import classification_report

pred_output = eval_trainer.predict(tokenized_ds["test"])
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

true_preds, true_labels = [], []
for pred, label in zip(preds, labels):
    tp, tl = [], []
    for p, l in zip(pred, label):
        if l != -100:
            tp.append(id2label[p])
            tl.append(id2label[l])
    true_preds.append(tp)
    true_labels.append(tl)

print(classification_report(true_labels, true_preds, digits=3))

              precision    recall  f1-score   support

       BRAND      0.811     0.834     0.823      1448
        GOOD      0.944     0.959     0.951      2259

   micro avg      0.892     0.910     0.901      3707
   macro avg      0.878     0.897     0.887      3707
weighted avg      0.892     0.910     0.901      3707



### Что показал разбор

По брендам precision 0.81 и recall 0.83 близко друг к другу. Значит модель
ошибается примерно поровну в обе стороны: иногда пропускает реальный бренд
(recall), иногда помечает брендом то, что им не является (precision). Ошибки сбалансированы, это просто предел
того, что модель выучила с этих данных.

Товары почти идеальны: precision 0.94, recall 0.96. Их модель находит уверенно.

Разрыв между macro avg (0.887) и weighted avg (0.901) небольшой, оба класса
вносят сопоставимый вклад, метрика не держится на одном за счёт другого.

Бренд труднее товара по понятным причинам: их тысячи разных, многие латиницей и
встречаются в обучении единично, а позиция часто содержит слова, похожие на бренд
(модели, артикулы), которые легко принять за него. 0.82 по такому открытому
классу нормальный результат.

## Примеры ошибок на брендах

Достаём позиции, где предсказанный бренд разошёлся с эталоном, и смотрим на текст.
Это показывает характер ошибок вживую: какие бренды пропущены, что ошибочно
принято за бренд.

In [15]:
test_tokens = dataset["test"]["tokens"]

def extract_brands(tokens, tags):
    brands, cur = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-BRAND":
            if cur: brands.append(" ".join(cur))
            cur = [tok]
        elif tag == "I-BRAND":
            cur.append(tok)
        else:
            if cur: brands.append(" ".join(cur)); cur = []
    if cur: brands.append(" ".join(cur))
    return brands

shown = 0
for i, (tp, tl) in enumerate(zip(true_preds, true_labels)):
    pred_b = extract_brands(test_tokens[i], tp)
    true_b = extract_brands(test_tokens[i], tl)
    if pred_b != true_b:
        print(f"текст:     {' '.join(test_tokens[i])}")
        print(f"эталон:    {true_b}")
        print(f"предсказ.: {pred_b}")
        print()
        shown += 1
    if shown >= 15:
        break

текст:     Арматура для бачка АНИпласт 2 - х кноп . хром бок / подв WC 9010 С
эталон:    []
предсказ.: ['АНИпласт']

текст:     Розетка 1 М СП " Севиль " UNIVersal
эталон:    ['Севиль']
предсказ.: []

текст:     РОЗЕТКА MAKEL MANOLYA С / У С / З БЕЛ
эталон:    ['MAKEL']
предсказ.: ['MAKEL MANOLYA']

текст:     506 Бензошланг ВАЗ ( 1 метр )
эталон:    []
предсказ.: ['ВАЗ']

текст:     Осетинский пирог с сыром ( 800 г )
эталон:    []
предсказ.: ['Осетинский']

текст:     9 / 12 Уголь активированный таб 250 мг № 10 4 р . за упак
эталон:    ['Уголь активированный']
предсказ.: []

текст:     Обувь ORTMANN LEON ортопедическая , арт . 7 . 11 . 2 р . 35 желтый / мятный микс , цветная подошва , soft
эталон:    ['ORTMANN']
предсказ.: ['ORTMANN LEON']

текст:     Китекат 800 г Курочка аппет
эталон:    []
предсказ.: ['Китекат']

текст:     Пирожное Шу со взбит . сливками ( Баттерфляй ) 0 . 28 кг * 251 . 00
эталон:    ['Баттерфляй']
предсказ.: []

текст:     СЫРОК ГЛАЗ ПРОСТОКВ КАКАО БЗМЖ МДЖ 23 % 

### Какие ошибки делает модель

Первое и самое частое это спорные случаи, где и человек засомневается. «АНИпласт»,
«ВАЗ», «Китекат», «USP», «SAVARTO» модель пометила как бренд, а в эталоне бренда
нет. Но это и правда похоже на бренды то есть модель размечает
разумно, просто разметчик ОФД счёл иначе.

Второе это границы многословных брендов. «MAKEL MANOLYA» вместо «MAKEL»,
«ORTMANN LEON» вместо «ORTMANN». Модель
правильно находит место бренда, но прихватывает лишнее слово или наоборот. Это
типичная для NER проблема границ сущности.

Третье это пропуски (recall): «Севиль», «Уголь активированный», «Баттерфляй»,
«Волжанка», «Умка», «Ремоколор» это реальные бренды, которые модель не увидела.
Часто это редкие бренды в кавычках или нетипичные названия, которых в обучении
было мало.

Главный вывод: чистых грубых ошибок мало. Большая часть расхождений это либо
спорная разметка (модель размечает логично, но иначе чем ОФД), либо границы
многословных брендов. Это объясняет, почему F1 по бренду 0.82, а не выше: потолок
тут частично в неоднозначности самой задачи, а не в слабости модели.

## Сохранение модели

Сохраняем выбранную модель (4 эпохи) и токенизатор в отдельную папку на Drive.

In [17]:
final_dir = "/content/drive/MyDrive/receipt-ai/ner_ru"
best_model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("Модель и токенизатор сохранены:", final_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель и токенизатор сохранены: /content/drive/MyDrive/receipt-ai/ner_ru
